# Phase 1: synthetic words

Composes images of handwritten Meitei Mayek words from TUMMHCD characters, and prepares
everything the recogniser (Phase 2) trains and is checked on:

1. **The paper's split** (`mayek.split` from the first project, pinned) and **character
   stores** for train / validation / test, without the test images that have a train twin
   and without label conflicts. Writes `results/glyph_store_stats.json`.
2. **How large each character is written**, recovered from stroke thickness (TUMMHCD
   stretched every character to 24 x 24). Writes `results/glyph_sizes_tummhcd.json`.
3. **The lexicon**: the Phase 0 word lists in everyday spelling (ꯢ written ꯏ), split by
   word into train / validation / test. Writes `results/lexicon_stats.json`.
4. **A contact sheet** of training words to look at, and a **spacing check** against real
   handwriting.
   Writes `results/spacing_synthetic_tummhcd.json`.
5. **Fixed synthetic validation and test sets** (5,000 words each), saved to Drive.
   Writes `results/synth_val.json` and `results/synth_test.json`.

Needs no GPU. Inputs on Drive: the TUMMHCD archive, and `WORK/wordlists/*.tsv` from the
Phase 0 notebook. Afterwards, send back (or commit) the files in `WORK/results` and the
two contact sheets in `WORK/synth`.

Character sizes are those measured on TUMMHCD in section 2 of this run (`--sizes font` would
give the font's). Optional secret (Colab key icon): `GITHUB_TOKEN`, only while the repository is private.

In [ ]:
# Where things are. Change these to match your Drive.
TUMMHCD_ZIP = "/content/drive/MyDrive/tummhcd98/TUMMHCD-TEST-TRAIN.zip"
WORK = "/content/drive/MyDrive/meitei-word-recognition"
WORDLISTS = f"{WORK}/wordlists"   # written by the Phase 0 notebook
REPO = "chingkheinganba231005/meitei-mayek-word-recognition"
BRANCHES = ["claude/intelligent-euler-rx42yl", "main"]  # the first that has the Phase 1 code is used
# the first project's package, for its split (15% of every train class for validation, seed 42)
MAYEK = ("git+https://github.com/chingkheinganba231005/Handwritten-Meitei-Mayek-Recognition"
         "@0d2c6e5b4c6589c635bf18048755137c1f3b519d")
N_FIXED = 5000      # words in each fixed synthetic set
SEED = 1

In [ ]:
import glob, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")  # only needed while the repository is private
except Exception:
    token = None

def run(*args):
    """Runs a command, shows its output, and stops the notebook if it fails."""
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token or "\0", "***")
    if out.strip():
        print(out)
    if r.returncode:
        raise SystemExit(f"exit code {r.returncode}: {' '.join(str(a) for a in args)[:200]}")

REPO_DIR = "/content/repo"
if not os.path.exists(f"{REPO_DIR}/mayek_words/synth.py"):
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    for branch in BRANCHES:
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", branch, url, REPO_DIR],
                           capture_output=True, text=True)
        if r.returncode == 0 and os.path.exists(f"{REPO_DIR}/mayek_words/synth.py"):
            break
    else:
        raise SystemExit("no branch in BRANCHES has the Phase 1 code. git: "
                         + r.stderr.replace(token or "\0", "***"))
    run("git", "-C", REPO_DIR, "remote", "set-url", "origin", f"https://github.com/{REPO}.git")
os.chdir(REPO_DIR)
run(sys.executable, "-m", "pip", "install", "-q", MAYEK, "scipy")
for d in ("results", "synth", "glyphs", "lexicon"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
run("git", "log", "-1", "--format=%h %s")

## 1. The paper's split and the character stores

In [ ]:
local = "/content/TUMMHCD-TEST-TRAIN.zip"
if not os.path.exists(local):
    shutil.copy(TUMMHCD_ZIP, local)
run(sys.executable, "-m", "mayek.split", "--zip", local, "--data-dir", "/content/data")
summary = json.load(open("/content/data/splits/summary.json"))
assert (summary["train"], summary["val"], summary["test"]) == (61504, 10826, 12794), f"not the paper's split: {summary}"
run(sys.executable, "scripts/build_glyphs.py", "--split-dir", "/content/data/splits",
    "--duplicates", "results/tummhcd_audit_duplicates.csv", "--out-dir", f"{WORK}/glyphs",
    "--stats", "results/glyph_store_stats.json")

## 2. How large each character is written

The method is checked first on the font's own characters, stretched the same way (the
answer is known there). `w` and `h` are in units of the letter height; `font` is the
printed size.

In [ ]:
run(sys.executable, "scripts/glyph_sizes.py", f"{WORK}/glyphs/train.npz", "--out", "results/glyph_sizes_tummhcd.json")

## 3. The lexicon

In [ ]:
lists = sorted(p for p in glob.glob(f"{WORDLISTS}/*.tsv") if not p.endswith("_consistent_i_lonsum.tsv"))
if not lists:
    raise SystemExit(f"no word lists in {WORDLISTS}: run the Phase 0 notebook first")
print("word lists:", [Path(p).name for p in lists])
run(sys.executable, "scripts/build_lexicon.py", *[f"{Path(p).stem}={p}" for p in lists],
    "--out-dir", f"{WORK}/lexicon", "--stats", "results/lexicon_stats.json")

## 4. Contact sheet

48 training words with the chosen settings. Look for signs in the wrong place or of the
wrong size, spacing, and anything a writer would never do.


In [ ]:
from IPython.display import Image as Show, display

sheet = f"{WORK}/synth/sheet.png"
run(sys.executable, "scripts/render_words.py", "--glyphs", f"{WORK}/glyphs/train.npz",
    "--lexicon", f"{WORK}/lexicon/train.tsv", "--n", 48, "--seed", SEED, "--sizes", "results/glyph_sizes_tummhcd.json", "--sheet", sheet)
display(Show(sheet))

### Spacing

Real handwriting joins about a third of neighbouring letters and leaves about a tenth of a
letter height between the others (six samples of published handwriting,
`results/spacing_web_samples.json`). Pages of synthetic words written with TUMMHCD
characters are measured the same way here.

In [ ]:
run(sys.executable, "scripts/measure_spacing.py", "--synthetic", 30, "--glyphs", f"{WORK}/glyphs/train.npz",
    "--lexicon", f"{WORK}/lexicon/train.tsv", "--sizes", "results/glyph_sizes_tummhcd.json", "--out", "results/spacing_synthetic_tummhcd.json")
print("real handwriting:", json.load(open("results/spacing_web_samples.json"))["median_over_samples"])

## 5. Fixed synthetic validation and test sets

Validation words are drawn from the validation lexicon and written with validation
characters, test words likewise. Rendered on the local disk, then saved to Drive as one
archive per set (many small files are slow on Drive).

In [ ]:
for split, seed in (("val", SEED), ("test", SEED + 1)):
    out = f"/content/synth/{split}"
    shutil.rmtree(out, ignore_errors=True)
    run(sys.executable, "scripts/render_words.py", "--glyphs", f"{WORK}/glyphs/{split}.npz",
        "--lexicon", f"{WORK}/lexicon/{split}.tsv", "--n", N_FIXED, "--seed", seed,
        "--sizes", "results/glyph_sizes_tummhcd.json", "--out-dir", out)
    shutil.copy(f"{out}/config.json", f"results/synth_{split}.json")
    shutil.make_archive(f"{WORK}/synth/{split}", "tar", out)
    print(f"{split}: {WORK}/synth/{split}.tar")

In [ ]:
for name in ("glyph_store_stats.json", "glyph_sizes_tummhcd.json", "lexicon_stats.json",
             "spacing_synthetic_tummhcd.json", "synth_val.json", "synth_test.json"):
    shutil.copy(f"results/{name}", f"{WORK}/results/")
print("Send back or commit:", f"{WORK}/results/*.json", "and", f"{WORK}/synth/sheet.png")